# SnowMart 出店戦略ハンズオン — Day 1
## データ基盤を構築する（60分）

### ストーリー
あなたはコンビニチェーン「**スノーマート**」のデータチームに配属された新メンバーです。  
スノーマートは全国に **500店舗** を展開しており、来年度中に **600店舗への拡大** を計画しています。  
今日のミッションは、出店戦略の土台となる **データ基盤をゼロから構築する** ことです。

### 今日やること
1. データベース作成 & CSVデータロード
2. Marketplace から天気データを取得
3. Time Travel & Zero-Copy Clone
4. Warehouse パフォーマンス体験
5. Dynamic Tables でパイプライン構築
6. Streamlit で可視化 + Cortex Code でブラッシュアップ

## Scene 1: 環境セットアップ（5分）
まずはデータベースとスキーマを作成します。

In [ ]:
-- ロールとウェアハウスの設定
USE ROLE ACCOUNTADMIN;
USE WAREHOUSE COMPUTE_WH;

In [ ]:
-- データベースとスキーマの作成
CREATE OR REPLACE DATABASE SNOWMART_DB;
CREATE OR REPLACE SCHEMA SNOWMART_DB.SNOWMART_SCHEMA;
USE SCHEMA SNOWMART_DB.SNOWMART_SCHEMA;

In [ ]:
-- データロード用の内部ステージを作成
CREATE OR REPLACE STAGE SNOWMART_STAGE
  DIRECTORY = (ENABLE = TRUE)
  FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

## Scene 2: データロード（12分）
スノーマートの業務データ5種類をロードします。
- 自社店舗マスタ（500店舗）
- 競合店舗（1,000店舗 — 場所と属性のみ。売上データは手に入らない）
- エリアマスタ（50エリア — 人口・年収・昼間人口比率）
- 日次売上（183,000行 — 500店舗×365日）
- 顧客レビュー（500件 — テキストデータ）

In [ ]:
-- テーブル作成
CREATE OR REPLACE TABLE SNOWMART_STORES (
    STORE_ID VARCHAR(10),
    STORE_NAME VARCHAR(100),
    PREFECTURE VARCHAR(20),
    CITY VARCHAR(50),
    LATITUDE FLOAT,
    LONGITUDE FLOAT,
    STORE_TYPE VARCHAR(10),
    FLOOR_AREA_SQM INT,
    OPEN_DATE DATE,
    NEAREST_STATION VARCHAR(100)
);

CREATE OR REPLACE TABLE COMPETITOR_STORES (
    COMPETITOR_ID VARCHAR(10),
    CHAIN_NAME VARCHAR(50),
    PREFECTURE VARCHAR(20),
    CITY VARCHAR(50),
    LATITUDE FLOAT,
    LONGITUDE FLOAT,
    FLOOR_AREA_SQM INT
);

CREATE OR REPLACE TABLE AREA_MASTER (
    AREA_ID VARCHAR(10),
    PREFECTURE VARCHAR(20),
    CITY VARCHAR(50),
    POPULATION INT,
    HOUSEHOLDS INT,
    DAYTIME_POPULATION_RATIO FLOAT,
    AVG_ANNUAL_INCOME INT,
    AREA_TYPE VARCHAR(20)
);

CREATE OR REPLACE TABLE DAILY_SALES (
    STORE_ID VARCHAR(10),
    SALES_DATE DATE,
    SALES_AMOUNT INT,
    CUSTOMER_COUNT INT,
    AVG_UNIT_PRICE FLOAT,
    FOOD_SALES INT,
    BEVERAGE_SALES INT,
    DAILY_GOODS_SALES INT
);

CREATE OR REPLACE TABLE CUSTOMER_REVIEWS (
    REVIEW_ID VARCHAR(10),
    STORE_ID VARCHAR(10),
    REVIEW_DATE DATE,
    RATING INT,
    REVIEW_TEXT VARCHAR(1000),
    REVIEWER_AGE_GROUP VARCHAR(20)
);

### ファイルアップロード
以下の手順でCSVファイルをステージにアップロードしてください：
1. Snowsight 左メニュー → **Data** → **Databases** → **SNOWMART_DB** → **SNOWMART_SCHEMA** → **Stages** → **SNOWMART_STAGE**
2. **「+ Files」** ボタンから以下のCSVファイルをアップロード：
   - `snowmart_stores.csv`
   - `competitor_stores.csv`
   - `area_master.csv`
   - `daily_sales.csv`
   - `customer_reviews.csv`

In [ ]:
-- COPY INTO でテーブルにロード
COPY INTO SNOWMART_STORES FROM @SNOWMART_STAGE/snowmart_stores.csv
  FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);
COPY INTO COMPETITOR_STORES FROM @SNOWMART_STAGE/competitor_stores.csv
  FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);
COPY INTO AREA_MASTER FROM @SNOWMART_STAGE/area_master.csv
  FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);
COPY INTO DAILY_SALES FROM @SNOWMART_STAGE/daily_sales.csv
  FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);
COPY INTO CUSTOMER_REVIEWS FROM @SNOWMART_STAGE/customer_reviews.csv
  FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

In [ ]:
-- 各テーブルの行数を確認
SELECT 'SNOWMART_STORES' AS TABLE_NAME, COUNT(*) AS ROW_COUNT FROM SNOWMART_STORES
UNION ALL SELECT 'COMPETITOR_STORES', COUNT(*) FROM COMPETITOR_STORES
UNION ALL SELECT 'AREA_MASTER', COUNT(*) FROM AREA_MASTER
UNION ALL SELECT 'DAILY_SALES', COUNT(*) FROM DAILY_SALES
UNION ALL SELECT 'CUSTOMER_REVIEWS', COUNT(*) FROM CUSTOMER_REVIEWS;

In [ ]:
-- 自社店舗の中身を確認
SELECT * FROM SNOWMART_STORES LIMIT 10;

In [ ]:
-- 都道府県別の店舗数
SELECT PREFECTURE, COUNT(*) AS STORE_COUNT
FROM SNOWMART_STORES
GROUP BY PREFECTURE
ORDER BY STORE_COUNT DESC
LIMIT 10;

## Scene 3: Marketplace 体験（8分）
出店戦略には天気データも重要です。雨の日は客足が減りますよね。  
このデータ、自分で集める必要はありません。**Snowflake Marketplace** にあります。

### Marketplace からデータを取得
1. Snowsight 左メニュー → **Marketplace**
2. 検索バーに **「Prepper Open Data Bank Japanese Weather」** と入力
3. **「Prepper Open Data Bank - Japanese Weather Data」** を選択
4. **「Get」** ボタンをクリック → データベース名はデフォルトのままで「Get」

> **ポイント**: Get ボタンを押すだけ。S3からダウンロードしてロードする作業は一切不要です。  
> これがSnowflakeのデータシェアリング。コピーすら作られません。

In [ ]:
-- Marketplace から取得した天気データベースを確認
SHOW DATABASES LIKE '%WEATHER%';

In [ ]:
-- 天気データの中身を確認
-- ※データベース名は取得時の名前に合わせてください
-- SELECT * FROM PREPPER_OPEN_DATA_BANK___JAPANESE_WEATHER_DATA.PUBLIC.JAPANESE_WEATHER_DATA LIMIT 10;

## Scene 4: Time Travel & Zero-Copy Clone（8分）
本番運用で一番怖いのは、データの誤削除。  
Snowflakeなら **過去に戻れます**。そして **一瞬で開発環境が作れます**。

In [ ]:
-- 現在の行数を確認
SELECT COUNT(*) AS BEFORE_COUNT FROM DAILY_SALES;

In [ ]:
-- 「誤って」10月以降の売上データを削除してしまった！
DELETE FROM DAILY_SALES WHERE SALES_DATE >= '2024-10-01';

In [ ]:
-- データが消えた！
SELECT COUNT(*) AS AFTER_DELETE FROM DAILY_SALES;

In [ ]:
-- Time Travel で120秒前のデータを確認
SELECT COUNT(*) AS RECOVERED_COUNT FROM DAILY_SALES AT(OFFSET => -120);

In [ ]:
-- テーブルを復旧する
CREATE OR REPLACE TABLE DAILY_SALES AS
SELECT * FROM DAILY_SALES AT(OFFSET => -120);

-- 復旧完了を確認
SELECT COUNT(*) AS RESTORED_COUNT FROM DAILY_SALES;

In [ ]:
-- 本番DBをまるごとクローン（開発環境として使う）
CREATE OR REPLACE DATABASE SNOWMART_DB_DEV CLONE SNOWMART_DB;

-- クローンの中身を確認（本番と同じ行数！）
SELECT COUNT(*) AS DEV_SALES_COUNT FROM SNOWMART_DB_DEV.SNOWMART_SCHEMA.DAILY_SALES;

**ポイント**
- Time Travel はデフォルトで過去1日分。Enterprise Edition なら90日まで拡張可能
- Clone は **メタデータのコピーだけ** なので、500GBのDBでも一瞬。ストレージ追加ゼロ
- AWSだと RDS スナップショット → リストアで数十分かかるものが、数秒です

## Scene 5: Warehouse パフォーマンス（5分）
Snowflakeはコンピュートとストレージが分離。  
ウェアハウスのサイズを変えるだけで、クエリが速くなります。

In [ ]:
USE SCHEMA SNOWMART_DB.SNOWMART_SCHEMA;

-- XS ウェアハウスで重めのクエリを実行
ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = 'XSMALL';

SELECT
    st.PREFECTURE,
    st.STORE_TYPE,
    DATE_TRUNC('MONTH', s.SALES_DATE) AS SALES_MONTH,
    COUNT(DISTINCT s.STORE_ID) AS STORE_COUNT,
    SUM(s.SALES_AMOUNT) AS TOTAL_SALES,
    SUM(s.CUSTOMER_COUNT) AS TOTAL_CUSTOMERS,
    AVG(s.AVG_UNIT_PRICE) AS AVG_PRICE,
    SUM(s.FOOD_SALES) AS TOTAL_FOOD,
    SUM(s.BEVERAGE_SALES) AS TOTAL_BEVERAGE,
    SUM(s.DAILY_GOODS_SALES) AS TOTAL_DAILY_GOODS
FROM DAILY_SALES s
JOIN SNOWMART_STORES st ON s.STORE_ID = st.STORE_ID
JOIN AREA_MASTER a ON st.PREFECTURE = a.PREFECTURE AND st.CITY = a.CITY
GROUP BY st.PREFECTURE, st.STORE_TYPE, DATE_TRUNC('MONTH', s.SALES_DATE)
ORDER BY TOTAL_SALES DESC;
-- → 実行時間を確認（Query Profile で確認）

In [ ]:
-- Medium にサイズアップして同じクエリを再実行
ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = 'MEDIUM';

SELECT
    st.PREFECTURE,
    st.STORE_TYPE,
    DATE_TRUNC('MONTH', s.SALES_DATE) AS SALES_MONTH,
    COUNT(DISTINCT s.STORE_ID) AS STORE_COUNT,
    SUM(s.SALES_AMOUNT) AS TOTAL_SALES,
    SUM(s.CUSTOMER_COUNT) AS TOTAL_CUSTOMERS,
    AVG(s.AVG_UNIT_PRICE) AS AVG_PRICE,
    SUM(s.FOOD_SALES) AS TOTAL_FOOD,
    SUM(s.BEVERAGE_SALES) AS TOTAL_BEVERAGE,
    SUM(s.DAILY_GOODS_SALES) AS TOTAL_DAILY_GOODS
FROM DAILY_SALES s
JOIN SNOWMART_STORES st ON s.STORE_ID = st.STORE_ID
JOIN AREA_MASTER a ON st.PREFECTURE = a.PREFECTURE AND st.CITY = a.CITY
GROUP BY st.PREFECTURE, st.STORE_TYPE, DATE_TRUNC('MONTH', s.SALES_DATE)
ORDER BY TOTAL_SALES DESC;
-- → XS のときより速い！

In [ ]:
-- コスト節約のためXSに戻す
ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = 'XSMALL';

**ポイント**
- サイズ変更はオンラインで即座。再起動不要
- XS→Mで4倍のコンピュートリソース
- AWSでいえば、EC2のインスタンスタイプをダウンタイムなしで変更できるイメージ
- Query Profile を開くと、どこに時間がかかったか可視化できます

## Scene 6: Dynamic Tables でパイプライン構築（8分）
分析用のサマリーテーブル、普通は ETL ジョブを書きますよね。  
Dynamic Tables なら **SELECT文を書くだけ** で自動更新パイプラインが完成します。

In [ ]:
-- 店舗 × エリア × 売上 を結合した分析用 Dynamic Table
CREATE OR REPLACE DYNAMIC TABLE STORE_SALES_ANALYSIS
    TARGET_LAG = '1 hour'
    WAREHOUSE = COMPUTE_WH
AS
SELECT
    s.STORE_ID,
    st.STORE_NAME,
    st.PREFECTURE,
    st.CITY,
    st.STORE_TYPE,
    st.FLOOR_AREA_SQM,
    st.NEAREST_STATION,
    a.POPULATION,
    a.DAYTIME_POPULATION_RATIO,
    a.AVG_ANNUAL_INCOME,
    a.AREA_TYPE,
    s.SALES_DATE,
    s.SALES_AMOUNT,
    s.CUSTOMER_COUNT,
    s.AVG_UNIT_PRICE,
    s.FOOD_SALES,
    s.BEVERAGE_SALES,
    s.DAILY_GOODS_SALES
FROM DAILY_SALES s
JOIN SNOWMART_STORES st ON s.STORE_ID = st.STORE_ID
LEFT JOIN AREA_MASTER a ON st.PREFECTURE = a.PREFECTURE AND st.CITY = a.CITY;

In [ ]:
-- Dynamic Table の状態を確認
SHOW DYNAMIC TABLES LIKE 'STORE_SALES_ANALYSIS';

In [ ]:
-- 分析テーブルを使ってクエリ
SELECT
    PREFECTURE,
    AREA_TYPE,
    COUNT(DISTINCT STORE_ID) AS STORE_COUNT,
    ROUND(AVG(SALES_AMOUNT)) AS AVG_DAILY_SALES,
    ROUND(AVG(CUSTOMER_COUNT)) AS AVG_DAILY_CUSTOMERS
FROM STORE_SALES_ANALYSIS
GROUP BY PREFECTURE, AREA_TYPE
ORDER BY AVG_DAILY_SALES DESC
LIMIT 20;

**ポイント**
- SELECT文を書いただけで、自動更新パイプラインが完成
- `TARGET_LAG = '1 hour'` = 元データが変わったら1時間以内に反映
- AWS Glue のジョブ定義 + CloudWatch スケジューラ + S3出力 をこの1文で代替

## Scene 7: Streamlit で可視化 + Cortex Code でブラッシュアップ（10分）
ここまでのデータを、ダッシュボードで可視化しましょう。  
Snowflake 上で直接 Streamlit アプリをデプロイし、AIの力でコードを改善します。

### Streamlit in Snowflake のデプロイ手順
1. Snowsight 左メニュー → **Projects** → **Streamlit**
2. **「+ Streamlit App」** をクリック
3. 設定:
   - App name: `SNOWMART_DASHBOARD`
   - Warehouse: `COMPUTE_WH`
   - Database: `SNOWMART_DB` / Schema: `SNOWMART_SCHEMA`
4. 「Create」をクリック
5. 以下のベースコードをエディタに貼り付けて **「Run」**

In [ ]:
# === このコードを Streamlit in Snowflake のエディタに貼り付けてください ===

import streamlit as st
from snowflake.snowpark.context import get_active_session

st.set_page_config(page_title="SnowMart Dashboard", layout="wide")
st.title("SnowMart 店舗分析ダッシュボード")

session = get_active_session()

# 都道府県別の売上サマリー
df_pref = session.sql("""
    SELECT
        PREFECTURE,
        COUNT(DISTINCT STORE_ID) AS STORE_COUNT,
        ROUND(AVG(SALES_AMOUNT)) AS AVG_DAILY_SALES,
        SUM(SALES_AMOUNT) AS TOTAL_SALES
    FROM STORE_SALES_ANALYSIS
    GROUP BY PREFECTURE
    ORDER BY TOTAL_SALES DESC
    LIMIT 15
""").to_pandas()

st.subheader("都道府県別 売上サマリー（上位15）")
st.bar_chart(df_pref.set_index("PREFECTURE")["TOTAL_SALES"])

# 月次トレンド
df_monthly = session.sql("""
    SELECT
        DATE_TRUNC('MONTH', SALES_DATE) AS MONTH,
        SUM(SALES_AMOUNT) AS TOTAL_SALES,
        SUM(CUSTOMER_COUNT) AS TOTAL_CUSTOMERS
    FROM STORE_SALES_ANALYSIS
    GROUP BY MONTH
    ORDER BY MONTH
""").to_pandas()

st.subheader("月次売上トレンド")
st.line_chart(df_monthly.set_index("MONTH")["TOTAL_SALES"])

### Cortex Code in Snowsight でブラッシュアップ
Streamlit エディタの **AI アシスタント**（Cortex）を使って改善しましょう。

**プロンプト例 1**: 「都道府県でフィルタリングできるセレクトボックスを追加してください」  
**プロンプト例 2**: 「エリアタイプ別の平均売上を横棒グラフで追加してください」  
**プロンプト例 3**: 「店舗タイプ（直営/FC）の売上比較をメトリクスカードで表示してください」

## Scene 8: Day 1 まとめ & Day 2 予告（4分）

### 今日やったこと
1. データベース作成 → CSVデータロード
2. Marketplace から天気データを即取得
3. Time Travel でデータ復旧、Clone で開発環境を一瞬で作成
4. Warehouse サイズ変更でパフォーマンスチューニング
5. Dynamic Tables で自動更新パイプライン
6. Streamlit + AI でダッシュボード構築

AWSで同等のことをやるなら RDS + S3 + Glue + Athena + Data Exchange + QuickSight + ... と複数サービスの組み合わせが必要です。

### Day 2 予告
次回はこのデータに **AI** を載せます。
- 顧客レビューの感情分析・要約
- 社内文書の自然言語検索（RAG）
- AIエージェントに **「どこに出店すべきか？」** を聞きます

## 付録: クリーンアップ
ハンズオン終了後にリソースを削除する場合は以下を実行してください。

In [ ]:
-- ※ Day 2 でもデータを使うため、今は実行しないでください！
-- DROP DATABASE IF EXISTS SNOWMART_DB;
-- DROP DATABASE IF EXISTS SNOWMART_DB_DEV;
-- ALTER WAREHOUSE COMPUTE_WH SET WAREHOUSE_SIZE = 'XSMALL';